# 1. File retrieval

In [5]:
import requests

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE4nnn/GSE4290/soft/GSE4290_family.soft.gz"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open("GSE4290_family.soft.gz", "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)

In [7]:
import GEOparse
import pandas as pd
import numpy as np

gse = GEOparse.get_GEO(
    filepath="../data/raw/geo/GSE4290_family.soft.gz",silent=True
)


c:\Users\haina\AppData\Local\Programs\Python\Python311\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


In [14]:
print(len(gse.gsms))      # samples
print(len(gse.gpls))      # platforms

180
1


# 2. Obtaining expression and metadata

## 2.1. Metadata

In [16]:
meta = gse.phenotype_data
meta.to_csv("../data/raw/geo/metadata_gse4290.csv")
meta.head()

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,taxid_ch1,...,contact_address,contact_city,contact_state,contact_zip/postal_code,contact_country,supplementary_file,relation,series_id,data_row_count,characteristics_ch1.0.Histopathological diagnostic:
GSM97793,HF0017_U133P2,GSM97793,Public on Apr 10 2006,Feb 22 2006,Aug 28 2018,RNA,1,Brain tissue from glioma patient,Homo sapiens,9606,...,9030 Old Georgetown Rd,Bethesda,MD,20892,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM97nn...,Reanalyzed by: GSE119087,GSE4290,54613,NaN
GSM97794,HF0024_U133P2,GSM97794,Public on Apr 10 2006,Feb 22 2006,Aug 28 2018,RNA,1,Brain tissue from glioma patient,Homo sapiens,9606,...,9030 Old Georgetown Rd,Bethesda,MD,20892,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM97nn...,Reanalyzed by: GSE119087,GSE4290,54613,NaN
GSM97795,HF0026_U133P2,GSM97795,Public on Apr 10 2006,Feb 22 2006,Aug 28 2018,RNA,1,Brain tissue from glioma patient,Homo sapiens,9606,...,9030 Old Georgetown Rd,Bethesda,MD,20892,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM97nn...,Reanalyzed by: GSE119087,GSE4290,54613,NaN
GSM97796,HF0031_U133P2,GSM97796,Public on Apr 10 2006,Feb 22 2006,Aug 28 2018,RNA,1,Brain tissue from glioma patient,Homo sapiens,9606,...,9030 Old Georgetown Rd,Bethesda,MD,20892,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM97nn...,Reanalyzed by: GSE119087,"GSE4290,GSE4536",54613,NaN
GSM97797,HF0050_U133P2,GSM97797,Public on Apr 10 2006,Feb 22 2006,Aug 28 2018,RNA,1,Brain tissue from glioma patient,Homo sapiens,9606,...,9030 Old Georgetown Rd,Bethesda,MD,20892,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM97nn...,Reanalyzed by: GSE119087,GSE4290,54613,NaN


### See types of samples

In [18]:
print(meta["characteristics_ch1.0.Histopathological diagnostic"].unique())

<StringArray>
[      'astrocytoma, grade 3',      'glioblastoma, grade 4',
 'oligodendroglioma, grade 2',                  'non-tumor',
 'oligodendroglioma, grade 3',                          nan,
       'astrocytoma, grade 2']
Length: 7, dtype: str


### Get ID of non-tumor and glioblastoma

In [44]:
healthy_id = meta[meta["characteristics_ch1.0.Histopathological diagnostic"] == "non-tumor"]["geo_accession"].tolist()
gbm_id = meta[meta["characteristics_ch1.0.Histopathological diagnostic"] == "glioblastoma, grade 4"]["geo_accession"].tolist()

print("Healthy samples:",len(healthy_id))
print("Glioblastoma samples:",len(gbm_id))

Healthy samples: 23
Glioblastoma samples: 77


## 2.2. Expression data

In [27]:
gse_expr = gse.pivot_samples("VALUE")
print(gse_expr.shape)
print(gse_expr.head(3))

(54613, 180)
name       GSM97793  GSM97794  GSM97795  GSM97796  GSM97797  GSM97798  \
ID_REF                                                                  
1007_s_at   10178.1   10122.9    7826.6   11098.4    8668.9    8659.2   
1053_at       388.2     517.5     352.4     609.9     430.1     592.5   
117_at        227.3     460.7     306.0     629.7     551.6     393.6   

name       GSM97799  GSM97800  GSM97801  GSM97802  ...  GSM97963  GSM97964  \
ID_REF                                             ...                       
1007_s_at    9267.2    4701.5   10702.4    6857.8  ...   11638.7   14652.2   
1053_at       378.9     282.7     355.7     735.4  ...    1229.4     424.0   
117_at        200.4     769.6     355.2     261.6  ...     245.4     199.8   

name       GSM97965  GSM97966  GSM97967  GSM97968  GSM97969  GSM97970  \
ID_REF                                                                  
1007_s_at   12849.9   11954.4    3929.9    6895.3   14618.3   10016.7   
1053_at    

### Apply log2 because values in the thousands

In [28]:
gse_expr = np.log2(gse_expr + 1)
print(gse_expr.head(3))

name        GSM97793   GSM97794   GSM97795   GSM97796   GSM97797   GSM97798  \
ID_REF                                                                        
1007_s_at  13.313322  13.305478  12.934354  13.438194  13.081800  13.080185   
1053_at     8.604368   9.018200   8.465158   9.254792   8.751879   9.213104   
117_at      7.834787   8.850812   8.262095   9.300810   9.110092   8.624247   

name        GSM97799   GSM97800   GSM97801   GSM97802  ...   GSM97963  \
ID_REF                                                 ...              
1007_s_at  13.178073  12.199212  13.385782  12.743740  ...  13.506766   
1053_at     8.569476   8.148222   8.478567   9.524346  ...  10.264912   
117_at      7.653920   9.589838   8.476544   8.036723  ...   7.944858   

name        GSM97964   GSM97965   GSM97966   GSM97967   GSM97968   GSM97969  \
ID_REF                                                                        
1007_s_at  13.838928  13.649582  13.545375  11.940644  12.751607  13.835587   
1

### Probe-gene mapping

In [40]:
gene_id = gse.gpls["GPL570"].table[["ID", "Gene Symbol"]].copy()



import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

import collapseprobes


gene_expr = collapseprobes.collapse_probes_to_genes(gse_expr,gene_id)
gene_expr


,GSM97793,GSM97794,GSM97795,GSM97796,GSM97797,GSM97798,GSM97799,GSM97800,GSM97801,GSM97802,...,GSM97963,GSM97964,GSM97965,GSM97966,GSM97967,GSM97968,GSM97969,GSM97970,GSM97971,GSM97972
Gene Symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,7.916477,7.608809,7.349613,6.244126,7.611025,6.882643,6.029011,7.448736,7.483816,7.013462,...,7.584211,6.663914,6.509379,7.635899,6.910493,7.577429,7.564531,6.877744,7.900263,7.223036
A1BG-AS1,4.566815,4.672425,4.626439,5.845490,4.738768,4.632268,4.972693,6.292782,5.083213,4.185867,...,4.035624,4.862947,3.906891,3.963474,5.149747,6.381975,4.263034,4.649615,4.314697,5.581954
A1CF,9.203593,8.189330,8.329124,8.083745,8.366322,8.432124,8.519243,8.429616,7.994353,8.074677,...,7.427103,8.332708,8.146187,8.222070,8.498650,8.248876,9.005344,8.761884,7.707359,7.847371
A2M,12.525129,13.292221,12.334608,14.357201,13.977468,12.993876,12.492254,11.269302,13.683905,12.953614,...,13.740350,12.312032,14.289515,14.151310,12.344573,11.931698,13.955468,12.368015,13.491164,12.954523
A2M-AS1,8.467606,7.787903,7.903279,7.351381,8.294161,7.416164,7.864186,7.671010,7.662490,7.356672,...,7.054197,7.237449,6.937815,7.180904,7.936638,7.761551,8.233140,8.526304,8.799605,8.824004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
hADV36S1,4.419539,6.922436,6.143638,6.451211,5.318317,7.207502,6.388878,6.416164,5.857981,6.726559,...,6.169925,5.274262,6.298292,6.274262,5.449561,6.542258,6.339850,6.787903,7.811856,5.472488
hsa-let-7a-3,8.710118,8.661422,8.391458,8.159871,8.466382,9.072267,9.860001,8.640245,8.842665,9.281466,...,9.083745,8.396177,8.174926,9.198936,7.924219,8.320124,8.745506,8.896635,9.566625,8.507398
hsa-let-7b,8.710118,8.661422,8.391458,8.159871,8.466382,9.072267,9.860001,8.640245,8.842665,9.281466,...,9.083745,8.396177,8.174926,9.198936,7.924219,8.320124,8.745506,8.896635,9.566625,8.507398


In [45]:
healthy_expr = gene_expr.loc[:, healthy_id]
gbm_expr = gene_expr.loc[:, gbm_id]

healthy_expr.to_csv("../data/interim/healthy_expr.csv")
gbm_expr.to_csv("../data/interim/gbm_expr.csv")

gbm_expr

,GSM97794,GSM97796,GSM97797,GSM97798,GSM97801,GSM97806,GSM97808,GSM97813,GSM97814,GSM97818,...,GSM97955,GSM97959,GSM97961,GSM97963,GSM97965,GSM97966,GSM97967,GSM97968,GSM97969,GSM97971
Gene Symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,7.608809,6.244126,7.611025,6.882643,7.483816,6.850499,6.984134,6.947199,7.947199,7.464342,...,8.108524,6.008989,7.869748,7.584211,6.509379,7.635899,6.910493,7.577429,7.564531,7.900263
A1BG-AS1,4.672425,5.845490,4.738768,4.632268,5.083213,4.145677,4.193772,4.292782,4.963474,4.802193,...,4.972693,7.217231,4.035624,4.035624,3.906891,3.963474,5.149747,6.381975,4.263034,4.314697
A1CF,8.189330,8.083745,8.366322,8.432124,7.994353,8.586089,7.840463,8.192293,8.155324,7.932510,...,7.627169,8.074677,7.985842,7.427103,8.146187,8.222070,8.498650,8.248876,9.005344,7.707359
A2M,13.292221,14.357201,13.977468,12.993876,13.683905,13.235446,12.849444,13.948960,12.636443,13.955023,...,13.244721,13.356961,12.990547,13.740350,14.289515,14.151310,12.344573,11.931698,13.955468,13.491164
A2M-AS1,7.787903,7.351381,8.294161,7.416164,7.662490,7.605109,7.139551,8.696620,6.920055,7.539934,...,7.216261,7.426265,7.188836,7.054197,6.937815,7.180904,7.936638,7.761551,8.233140,8.799605
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
hADV36S1,6.922436,6.451211,5.318317,7.207502,5.857981,6.339850,6.361066,6.958843,6.821455,6.261155,...,6.921246,6.072535,4.459432,6.169925,6.298292,6.274262,5.449561,6.542258,6.339850,7.811856
hsa-let-7a-3,8.661422,8.159871,8.466382,9.072267,8.842665,9.280307,9.325305,8.343408,8.807677,8.623881,...,8.896332,8.483413,8.567956,9.083745,8.174926,9.198936,7.924219,8.320124,8.745506,9.566625
hsa-let-7b,8.661422,8.159871,8.466382,9.072267,8.842665,9.280307,9.325305,8.343408,8.807677,8.623881,...,8.896332,8.483413,8.567956,9.083745,8.174926,9.198936,7.924219,8.320124,8.745506,9.566625
